# Montar Drive e instalar dependencias

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!nvidia-smi

Wed Apr 22 06:58:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:04:00.0 Off |                    0 |
| N/A   30C    P0             70W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!pip install nnunetv2 nibabel scikit-image trimesh scipy pycpd pandas tqdm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.6/205.6 kB 7.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 11.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.8/740.8 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 136.5 MB/s eta 0

# Configurar rutas

In [ ]:
import os
from pathlib import Path

# ── AJUSTA ESTA RUTA ────────────────────────────────────────
DRIVE_VERSE_DIR = '/content/drive/MyDrive/VerSe_2020_Dataset'
# ────────────────────────────────────────────────────────────

BASE_DIR        = '/content/nnunet_verse'
VERSE_EXTRACTED = f'{BASE_DIR}/verse_extracted'
NNUNET_RAW      = f'{BASE_DIR}/nnUNet_raw'
NNUNET_PREPROC  = f'{BASE_DIR}/nnUNet_preprocessed'
NNUNET_RESULTS  = f'{BASE_DIR}/nnUNet_results'

DRIVE_CHECKPOINTS = f'{DRIVE_VERSE_DIR}/checkpoints'
DRIVE_RESULTS     = f'{DRIVE_VERSE_DIR}/results'

for d in [BASE_DIR, VERSE_EXTRACTED, NNUNET_RAW, NNUNET_PREPROC,
          NNUNET_RESULTS, DRIVE_CHECKPOINTS, DRIVE_RESULTS]:
    os.makedirs(d, exist_ok=True)

os.environ['nnUNet_raw']          = NNUNET_RAW
os.environ['nnUNet_preprocessed'] = NNUNET_PREPROC
os.environ['nnUNet_results']      = NNUNET_RESULTS

DATASET_ID   = 507
DATASET_NAME = f'Dataset{DATASET_ID:03d}_VerSe2020'
RANDOM_SEED  = 42

VERSE_LABEL_TO_NAME = {
    1:'C1',  2:'C2',  3:'C3',  4:'C4',  5:'C5',  6:'C6',  7:'C7',
    8:'T1',  9:'T2',  10:'T3', 11:'T4', 12:'T5', 13:'T6',
    14:'T7', 15:'T8', 16:'T9', 17:'T10',18:'T11',19:'T12',
    20:'L1', 21:'L2', 22:'L3', 23:'L4', 24:'L5', 25:'L6', 26:'S1',
    28:'T13',
}
ALL_LABELS  = sorted(VERSE_LABEL_TO_NAME.keys())
LABEL_MAP   = {verse_l: cont_l for cont_l, verse_l in enumerate(ALL_LABELS, start=1)}
INVERSE_MAP = {v: k for k, v in LABEL_MAP.items()}

print('✓ Configuración cargada')
print(f'  Drive VerSe:         {DRIVE_VERSE_DIR}')
print(f'  nnUNet_raw:          {NNUNET_RAW}')
print(f'  nnUNet_preprocessed: {NNUNET_PREPROC}')

✓ Configuración cargada
  Drive VerSe:         /content/drive/MyDrive/VerSe_2020_Dataset
  nnUNet_raw:          /content/nnunet_verse/nnUNet_raw
  nnUNet_preprocessed: /content/nnunet_verse/nnUNet_preprocessed


# Descomprimir ZIPs

In [ ]:
import zipfile
from pathlib import Path

zips_to_extract = {
    'train':                f'{DRIVE_VERSE_DIR}/01_training.zip',
    'val':                  f'{DRIVE_VERSE_DIR}/02_validation.zip',
    'test':                 f'{DRIVE_VERSE_DIR}/03_test.zip',
    'dataset-verse19training':   f'{DRIVE_VERSE_DIR}/dataset-verse19training.zip',
    'dataset-verse19validation': f'{DRIVE_VERSE_DIR}/dataset-verse19validation.zip',
    'dataset-verse19test':       f'{DRIVE_VERSE_DIR}/dataset-verse19test.zip',
}

for split_name, zip_path in zips_to_extract.items():
    if not os.path.exists(zip_path):
        print(f'⚠ No encontrado: {zip_path}')
        continue

    out_dir = f'{VERSE_EXTRACTED}/{split_name}'
    if os.path.exists(out_dir) and len(os.listdir(out_dir)) > 0:
        print(f'✓ {split_name} ya descomprimido')
        continue

    print(f'Descomprimiendo {split_name}...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(out_dir)
    print(f'  ✓ {split_name} listo')

# Verificar
print('\n--- Verificando ---')
for split_name in zips_to_extract.keys():
    split_dir = Path(f'{VERSE_EXTRACTED}/{split_name}')
    if not split_dir.exists():
        continue
    cts  = list(split_dir.rglob('*_ct.nii*'))
    segs = list(split_dir.rglob('*_seg-vert_msk.nii*'))
    print(f'  {split_name}: {len(cts)} CTs, {len(segs)} máscaras')

Descomprimiendo train...
  ✓ train listo
Descomprimiendo val...
  ✓ val listo
Descomprimiendo test...
  ✓ test listo
Descomprimiendo dataset-verse19training...
  ✓ dataset-verse19training listo
Descomprimiendo dataset-verse19validation...
  ✓ dataset-verse19validation listo
Descomprimiendo dataset-verse19test...
  ✓ dataset-verse19test listo

--- Verificando ---
  train: 61 CTs, 61 máscaras
  val: 80 CTs, 80 máscaras
  test: 73 CTs, 73 máscaras
  dataset-verse19training: 80 CTs, 80 máscaras
  dataset-verse19validation: 40 CTs, 40 máscaras
  dataset-verse19test: 40 CTs, 40 máscaras


# Recopilar casos

In [ ]:
def collect_verse_cases(split_name):
    split_dir   = Path(VERSE_EXTRACTED) / split_name
    rawdata_dir = None
    deriv_dir   = None

    for d in split_dir.rglob('rawdata'):
        rawdata_dir = d; break
    for d in split_dir.rglob('derivatives'):
        deriv_dir = d; break

    if rawdata_dir is None or deriv_dir is None:
        print(f'  ⚠ No encontrado rawdata/derivatives en {split_dir}')
        return []

    cases = []
    for patient_dir in sorted(rawdata_dir.iterdir()):
        if not patient_dir.is_dir():
            continue
        pid = patient_dir.name

        deriv_patient = deriv_dir / pid
        if not deriv_patient.exists():
            continue

        ct_files  = list(patient_dir.glob('*_ct.nii*'))
        seg_files = list(deriv_patient.glob('*_seg-vert_msk.nii*'))
        ctd_files = list(deriv_patient.glob('*_seg-subreg_ctd*')) + \
                    list(deriv_patient.glob('*_seg-vb_ctd*'))

        for ct_file in ct_files:
            prefix    = ct_file.name.replace('_ct.nii.gz', '').replace('_ct.nii', '')
            seg_match = [s for s in seg_files if prefix in s.name]
            ctd_match = [c for c in ctd_files if prefix in c.name]

            if seg_match:
                cases.append({
                    'patient_id': pid,
                    'ct_path':    ct_file,
                    'seg_path':   seg_match[0],
                    'ctd_path':   ctd_match[0] if ctd_match else None,
                    'split':      split_name,
                })

    print(f'  {split_name}: {len(cases)} casos')
    return cases

print('Recopilando casos...')
train_cases      = collect_verse_cases('train')
val_cases        = collect_verse_cases('val')
test_cases       = collect_verse_cases('test')
train_cases_2019 = collect_verse_cases('dataset-verse19training')
val_cases_2019   = collect_verse_cases('dataset-verse19validation')
test_cases_2019  = collect_verse_cases('dataset-verse19test')

trainval_cases = train_cases + val_cases + train_cases_2019 + val_cases_2019
test_cases_all = test_cases + test_cases_2019

print(f'\nTrain+Val: {len(trainval_cases)} | Test: {len(test_cases_all)}')
print(f'Total:     {len(trainval_cases) + len(test_cases_all)} CTs')

Recopilando casos...
  train: 61 casos
  val: 80 casos
  test: 73 casos
  dataset-verse19training: 80 casos
  dataset-verse19validation: 40 casos
  dataset-verse19test: 40 casos

Train+Val: 261 | Test: 113
Total:     374 CTs


# Convertir al formato nnU-Net

In [ ]:
import json
import numpy as np
import nibabel as nib
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor

def verify_affine_alignment(img_nib, seg_nib):
    return np.allclose(img_nib.affine, seg_nib.affine, atol=1e-3)

def remap_segmentation(seg_array):
    new_seg = np.zeros_like(seg_array, dtype=np.uint8)
    for verse_label, cont_label in LABEL_MAP.items():
        new_seg[seg_array == verse_label] = cont_label
    return new_seg

dataset_dir = Path(NNUNET_RAW) / DATASET_NAME
images_tr   = dataset_dir / 'imagesTr'
labels_tr   = dataset_dir / 'labelsTr'
images_ts   = dataset_dir / 'imagesTs'
labels_ts   = dataset_dir / 'labelsTs'

for d in [images_tr, labels_tr, images_ts, labels_ts]:
    d.mkdir(parents=True, exist_ok=True)

def process_single_case(args):
    i, case, images_dir, labels_dir, prefix = args
    case_id = f'{prefix}_{i+1:04d}'

    out_img = images_dir / f'{case_id}_0000.nii.gz'
    out_seg = labels_dir / f'{case_id}.nii.gz'

    if out_img.exists() and out_seg.exists():
        return case_id, None

    img_nib = nib.load(str(case['ct_path']))
    seg_nib = nib.load(str(case['seg_path']))

    issue = None
    if not verify_affine_alignment(img_nib, seg_nib):
        issue = case['patient_id']

    nib.save(img_nib, str(out_img))

    seg_data     = np.asarray(seg_nib.dataobj).astype(np.uint8)
    seg_remapped = remap_segmentation(seg_data)
    new_seg = nib.Nifti1Image(
        seg_remapped,
        affine=seg_nib.affine,
        header=seg_nib.header
    )
    new_seg.header.set_data_dtype(np.uint8)
    nib.save(new_seg, str(out_seg))

    return case_id, issue

def process_cases(cases, images_dir, labels_dir, prefix, n_workers=16):
    ids              = []
    alignment_issues = []
    args_list        = [(i, case, images_dir, labels_dir, prefix)
                        for i, case in enumerate(cases)]

    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        futures = [executor.submit(process_single_case, args) for args in args_list]
        results = []
        with tqdm(total=len(futures), desc=f'Procesando {prefix}',
                  unit='CT', bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]') as pbar:
            for future in futures:
                result = future.result()
                results.append(result)
                pbar.update(1)

    for case_id, issue in results:
        ids.append(case_id)
        if issue:
            alignment_issues.append(issue)

    return ids, alignment_issues

print('Procesando train+val...')
train_ids, issues_tr = process_cases(trainval_cases, images_tr, labels_tr, 'VerSe', n_workers=16)

print('Procesando test...')
test_ids, issues_ts  = process_cases(test_cases_all, images_ts, labels_ts, 'VerSe_test', n_workers=16)

all_issues = issues_tr + issues_ts
if all_issues:
    print(f'⚠ Affine desalineada en: {all_issues}')

print(f'\n✓ {len(train_ids)} train+val, {len(test_ids)} test')
print(f'  Total: {len(train_ids) + len(test_ids)} CTs')

Procesando train+val...


Procesando VerSe:   0%|          | 0/261 [00:00<?, ?CT/s]

Procesando test...


Procesando VerSe_test:   0%|          | 0/113 [00:00<?, ?CT/s]


✓ 261 train+val, 113 test
  Total: 374 CTs


# Corregir warnings de affine

In [ ]:
from tqdm.notebook import tqdm

dataset_dir = Path(NNUNET_RAW) / DATASET_NAME

splits = [
    (dataset_dir / 'imagesTr', dataset_dir / 'labelsTr', 'Train'),
    (dataset_dir / 'imagesTs', dataset_dir / 'labelsTs', 'Test'),
]

total_fixed = []

for images_dir, labels_dir, split_name in splits:
    print(f'\n=== {split_name} ===')
    seg_files = sorted(labels_dir.glob('*.nii.gz'))
    fixed     = []

    for seg_path in tqdm(seg_files, desc=f'Corrigiendo {split_name}'):
        case_id  = seg_path.stem.replace('.nii', '')
        img_path = images_dir / f'{case_id}_0000.nii.gz'

        if not img_path.exists():
            continue

        img_nib = nib.load(str(img_path))
        seg_nib = nib.load(str(seg_path))

        needs_fix = False

        if not np.allclose(img_nib.affine, seg_nib.affine, atol=1e-3):
            needs_fix = True

        if not np.allclose(
            np.abs(np.diag(img_nib.affine)[:3]),
            np.abs(np.diag(seg_nib.affine)[:3]), atol=1e-4
        ):
            needs_fix = True

        if needs_fix:
            seg_data = np.asarray(seg_nib.dataobj).astype(np.uint8)
            new_seg  = nib.Nifti1Image(seg_data, img_nib.affine, img_nib.header)
            new_seg.header.set_data_dtype(np.uint8)
            nib.save(new_seg, str(seg_path))
            fixed.append(case_id)

    print(f'  ✓ Corregidos: {len(fixed)}')
    total_fixed += fixed

print(f'\n✓ Total corregidos: {len(total_fixed)}')
print('Ahora sí ejecuta el preprocessing')


=== Train ===


Corrigiendo Train:   0%|          | 0/261 [00:00<?, ?it/s]

  ✓ Corregidos: 0

=== Test ===


Corrigiendo Test:   0%|          | 0/113 [00:00<?, ?it/s]

  ✓ Corregidos: 0

✓ Total corregidos: 0
Ahora sí ejecuta el preprocessing


# Crear dataset.json

In [ ]:
labels_dict = {'background': 0}
for cont_label in sorted(INVERSE_MAP.keys()):
    verse_label = INVERSE_MAP[cont_label]
    labels_dict[VERSE_LABEL_TO_NAME[verse_label]] = cont_label

dataset_json = {
    'channel_names': {'0': 'CT'},
    'labels':        labels_dict,
    'numTraining':   len(train_ids),
    'file_ending':   '.nii.gz',
    'name':          DATASET_NAME,
    'description':   'VerSe 2019+2020 — 355 pacientes, 374 CTs',
    'reference':     'Sekuboyina et al., Medical Image Analysis 73, 102166 (2021)',
}

with open(dataset_dir / 'dataset.json', 'w') as f:
    json.dump(dataset_json, f, indent=2)

splits_info = {
    'random_seed':   RANDOM_SEED,
    'train_val_ids': train_ids,
    'test_ids':      test_ids,
    'label_map':     {str(k): v for k, v in LABEL_MAP.items()},
}
with open(dataset_dir / 'splits_info.json', 'w') as f:
    json.dump(splits_info, f, indent=2)

print(f'✓ dataset.json creado con {len(labels_dict)-1} clases vertebrales')

✓ dataset.json creado con 27 clases vertebrales


# Eliminar los ZIPs antes del Preprocessing para liberar almacenamiento en disco

In [ ]:
import shutil
from pathlib import Path

print('Liberando espacio...')

splits = [
    'train',
    'val',
    'test',
    'dataset-verse19training',
    'dataset-verse19validation',
    'dataset-verse19test',
]

for split in splits:
    split_dir = Path(VERSE_EXTRACTED) / split
    if split_dir.exists():
        size = sum(f.stat().st_size for f in split_dir.rglob('*') if f.is_file())
        size_gb = round(size / 1e9, 2)
        shutil.rmtree(str(split_dir))
        print(f'  ✓ {split} eliminado ({size_gb} GB liberados)')
    else:
        print(f'  - {split} no existe, saltando')

# Verificar espacio libre
import shutil as sh
total, used, free = sh.disk_usage('/')
print(f'\nEspacio libre ahora: {round(free / 1e9, 1)} GB')
print(f'Espacio usado:       {round(used / 1e9, 1)} GB')

Liberando espacio...
  - train no existe, saltando
  - val no existe, saltando
  - test no existe, saltando
  - dataset-verse19training no existe, saltando
  - dataset-verse19validation no existe, saltando
  - dataset-verse19test no existe, saltando

Espacio libre ahora: 123.9 GB
Espacio usado:       129.2 GB


In [ ]:
import nibabel as nib
import numpy as np
from pathlib import Path

dataset_dir = Path(NNUNET_RAW) / DATASET_NAME
cases_to_fix = ['VerSe_0049', 'VerSe_0082']

for case_id in cases_to_fix:
    img_path = dataset_dir / 'imagesTr' / f'{case_id}_0000.nii.gz'
    seg_path = dataset_dir / 'labelsTr' / f'{case_id}.nii.gz'

    img_nib = nib.load(str(img_path))
    seg_nib = nib.load(str(seg_path))

    print(f'{case_id}:')
    print(f'  Spacing img: {np.abs(np.diag(img_nib.affine)[:3])}')
    print(f'  Spacing seg: {np.abs(np.diag(seg_nib.affine)[:3])}')

    seg_data = np.asarray(seg_nib.dataobj).astype(np.uint8)
    new_seg  = nib.Nifti1Image(seg_data, img_nib.affine, img_nib.header)
    new_seg.header.set_data_dtype(np.uint8)
    nib.save(new_seg, str(seg_path))
    print(f'  ✓ Corregido')

print('\n✓ Listo, ejecuta el preprocessing de nuevo')

VerSe_0049:
  Spacing img: [0.76171875 0.76171875 0.59997559]
  Spacing seg: [0.76171875 0.76171875 0.60000002]
  ✓ Corregido
VerSe_0082:
  Spacing img: [0.9765625  0.9765625  0.59999084]
  Spacing seg: [0.9765625  0.9765625  0.60000002]
  ✓ Corregido

✓ Listo, ejecuta el preprocessing de nuevo


# Preprocessing

In [ ]:
import subprocess

cmd = [
    'nnUNetv2_plan_and_preprocess',
    '-d', str(DATASET_ID),
    '--verify_dataset_integrity',
    '-c', '2d', '3d_fullres', '3d_lowres',
    '-np', '8',
    '--verbose',
]

print('Ejecutando plan_and_preprocess...')
print('='*60)

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Sin filtrar — mostrar TODO el output para ver qué casos fallan
for line in process.stdout:
    print(line, end='', flush=True)

process.wait()

print('='*60)
print('\n✓ Preprocessing completado' if process.returncode == 0
      else f'\n✗ Error: {process.returncode}')

Se truncaron las últimas líneas 5000 del resultado de transmisión.
10 10000
11 10000
12 10000
13 10000
14 10000
15 10000
16 10000
17 10000
18 10000
19 10000
/content/nnunet_verse/nnUNet_raw/Dataset507_VerSe2020/labelsTr/VerSe_0206.nii.gz
old shape: (68, 568, 216), new_shape: [136 582 221], old_spacing: [np.float64(2.0003225803375244), np.float64(1.0), np.float64(1.0)], new_spacing: [1.0, 0.9765625, 0.9765625], fn_data: functools.partial(<function resample_data_or_seg_to_shape at 0x78a822cb5e40>, is_seg=False, order=3, order_z=0, force_separate_z=None)
7 10000
8 10000
9 10000
10 10000
11 10000
12 10000
13 10000
14 10000
15 10000
16 10000
17 10000
18 10000
19 10000
20 10000
21 10000
22 10000
23 10000
24 10000
25 10000
/content/nnunet_verse/nnUNet_raw/Dataset507_VerSe2020/labelsTr/VerSe_0209.nii.gz
old shape: (61, 198, 114), new_shape: [122 203 117], old_spacing: [np.float64(2.0), np.float64(1.0), np.float64(1.0)], new_spacing: [1.0, 0.9765625, 0.9765625], fn_data: functools.partial(<func

# Guardar preprocessing en Drive

In [ ]:
import shutil
from pathlib import Path

DRIVE_BASE = '/content/drive/MyDrive/preprocessed_verse_for_training'
Path(DRIVE_BASE).mkdir(parents=True, exist_ok=True)
print(f'Guardando en: {DRIVE_BASE}')

# Preprocessing completo
src = Path('/content/nnunet_verse/nnUNet_preprocessed/Dataset507_VerSe2020')
dst = Path(DRIVE_BASE) / 'nnUNet_preprocessed' / 'Dataset507_VerSe2020'
dst.mkdir(parents=True, exist_ok=True)
shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
print('✓ nnUNet_preprocessed → Drive')

# imagesTs
src2 = Path('/content/nnunet_verse/nnUNet_raw/Dataset507_VerSe2020/imagesTs')
dst2 = Path(DRIVE_BASE) / 'nnUNet_raw' / 'Dataset507_VerSe2020' / 'imagesTs'
dst2.mkdir(parents=True, exist_ok=True)
shutil.copytree(str(src2), str(dst2), dirs_exist_ok=True)
print('✓ imagesTs → Drive')

# labelsTs
src3 = Path('/content/nnunet_verse/nnUNet_raw/Dataset507_VerSe2020/labelsTs')
dst3 = Path(DRIVE_BASE) / 'nnUNet_raw' / 'Dataset507_VerSe2020' / 'labelsTs'
dst3.mkdir(parents=True, exist_ok=True)
shutil.copytree(str(src3), str(dst3), dirs_exist_ok=True)
print('✓ labelsTs → Drive')

# JSONs
for json_file in ['dataset.json', 'splits_info.json']:
    src4 = Path('/content/nnunet_verse/nnUNet_raw/Dataset507_VerSe2020') / json_file
    dst4 = Path(DRIVE_BASE) / 'nnUNet_raw' / 'Dataset507_VerSe2020' / json_file
    if src4.exists():
        shutil.copy2(str(src4), str(dst4))
        print(f'✓ {json_file} → Drive')

print(f'\n✓ Todo guardado en: {DRIVE_BASE}')

Guardando en: /content/drive/MyDrive/preprocessed_verse_for_training
✓ nnUNet_preprocessed → Drive
✓ imagesTs → Drive
✓ labelsTs → Drive
✓ dataset.json → Drive
✓ splits_info.json → Drive

✓ Todo guardado en: /content/drive/MyDrive/preprocessed_verse_for_training


# Verificaciones extras

In [ ]:
import os
from pathlib import Path

DRIVE_BASE = Path('/content/drive/MyDrive/preprocessed_verse_for_training')

total_size = 0
for f in DRIVE_BASE.rglob('*'):
    if f.is_file():
        total_size += f.stat().st_size

print(f'Carpeta: {DRIVE_BASE}')
print(f'Tamaño total: {round(total_size / 1e9, 2)} GB')

# Desglose por subcarpeta
print('\nDesglose:')
for subdir in sorted(DRIVE_BASE.iterdir()):
    if subdir.is_dir():
        sub_size = sum(f.stat().st_size for f in subdir.rglob('*') if f.is_file())
        print(f'  {subdir.name}: {round(sub_size / 1e9, 2)} GB')

Carpeta: /content/drive/MyDrive/preprocessed_verse_for_training
Tamaño total: 67.19 GB

Desglose:
  nnUNet_preprocessed: 49.07 GB
  nnUNet_raw: 18.12 GB


In [ ]:
from pathlib import Path

DRIVE_BASE = Path('/content/drive/MyDrive/preprocessed_verse_for_training')

print('=== Verificando integridad ===\n')

# Carpetas esperadas
expected = {
    'nnUNet_preprocessed/Dataset507_VerSe2020/gt_segmentations':    'gt_segmentations',
    'nnUNet_preprocessed/Dataset507_VerSe2020/nnUNetPlans_2d':      '2D preprocessing',
    'nnUNet_preprocessed/Dataset507_VerSe2020/nnUNetPlans_3d_fullres': '3D fullres preprocessing',
    'nnUNet_preprocessed/Dataset507_VerSe2020/nnUNetPlans_3d_lowres':  '3D lowres preprocessing',
    'nnUNet_preprocessed/Dataset507_VerSe2020/nnUNetPlans.json':     'Plans config',
    'nnUNet_preprocessed/Dataset507_VerSe2020/dataset_fingerprint.json': 'Fingerprint',
    'nnUNet_raw/Dataset507_VerSe2020/imagesTs':   'imagesTs',
    'nnUNet_raw/Dataset507_VerSe2020/labelsTs':   'labelsTs',
    'nnUNet_raw/Dataset507_VerSe2020/dataset.json':    'dataset.json',
    'nnUNet_raw/Dataset507_VerSe2020/splits_info.json': 'splits_info.json',
}

all_ok = True
for rel_path, name in expected.items():
    full_path = DRIVE_BASE / rel_path
    exists = full_path.exists()
    status = '✓' if exists else '✗'
    if not exists:
        all_ok = False
    print(f'  {status} {name}')

# Contar archivos en preprocessing
print('\n=== Conteo de archivos ===')
for config in ['nnUNetPlans_2d', 'nnUNetPlans_3d_fullres', 'nnUNetPlans_3d_lowres']:
    config_path = DRIVE_BASE / 'nnUNet_preprocessed' / 'Dataset507_VerSe2020' / config
    if config_path.exists():
        n_files = len(list(config_path.rglob('*')))
        print(f'  {config}: {n_files} archivos')

n_images = len(list((DRIVE_BASE / 'nnUNet_raw/Dataset507_VerSe2020/imagesTs').glob('*.nii.gz')))
n_labels = len(list((DRIVE_BASE / 'nnUNet_raw/Dataset507_VerSe2020/labelsTs').glob('*.nii.gz')))
print(f'  imagesTs: {n_images} CTs')
print(f'  labelsTs: {n_labels} máscaras')

print(f'\n{"✓ Todo OK — puedes desconectar" if all_ok else "✗ Faltan archivos — NO desconectes"}')

=== Verificando integridad ===

  ✓ gt_segmentations
  ✓ 2D preprocessing
  ✓ 3D fullres preprocessing
  ✓ 3D lowres preprocessing
  ✓ Plans config
  ✓ Fingerprint
  ✓ imagesTs
  ✓ labelsTs
  ✓ dataset.json
  ✓ splits_info.json

=== Conteo de archivos ===
  nnUNetPlans_2d: 783 archivos
  nnUNetPlans_3d_fullres: 783 archivos
  nnUNetPlans_3d_lowres: 783 archivos
  imagesTs: 113 CTs
  labelsTs: 113 máscaras

✓ Todo OK — puedes desconectar


# Comprobar tamaño de preprocessed

In [ ]:
import subprocess

DRIVE_BASE = '/content/drive/MyDrive/VerSe_2020_Dataset/preprocessed_verse_for_training'

print('Calculando tamaños en Drive...\n')

# Tamaño total de la carpeta
result = subprocess.run(['du', '-sh', DRIVE_BASE], capture_output=True, text=True)
print(f'Total preprocessed_verse_for_training: {result.stdout.split()[0]}')

print('\nDesglose por subcarpeta:')

# nnUNet_preprocessed
result = subprocess.run(
    ['du', '-sh', f'{DRIVE_BASE}/nnUNet_preprocessed'],
    capture_output=True, text=True
)
print(f'  nnUNet_preprocessed: {result.stdout.split()[0]}')

# Desglose interno del preprocessed
for config in ['nnUNetPlans_2d', 'nnUNetPlans_3d_fullres', 'nnUNetPlans_3d_lowres', 'gt_segmentations']:
    result = subprocess.run(
        ['du', '-sh', f'{DRIVE_BASE}/nnUNet_preprocessed/Dataset507_VerSe2020/{config}'],
        capture_output=True, text=True
    )
    print(f'    {config}: {result.stdout.split()[0]}')

# nnUNet_raw
result = subprocess.run(
    ['du', '-sh', f'{DRIVE_BASE}/nnUNet_raw'],
    capture_output=True, text=True
)
print(f'  nnUNet_raw: {result.stdout.split()[0]}')

# Desglose interno del raw
for folder in ['imagesTs', 'labelsTs']:
    result = subprocess.run(
        ['du', '-sh', f'{DRIVE_BASE}/nnUNet_raw/Dataset507_VerSe2020/{folder}'],
        capture_output=True, text=True
    )
    print(f'    {folder}: {result.stdout.split()[0]}')

Calculando tamaños en Drive...

Total preprocessed_verse_for_training: 63G

Desglose por subcarpeta:
  nnUNet_preprocessed: 46G
    nnUNetPlans_2d: 22G
    nnUNetPlans_3d_fullres: 21G
    nnUNetPlans_3d_lowres: 4.2G
    gt_segmentations: 143M
  nnUNet_raw: 17G
    imagesTs: 17G
    labelsTs: 76M


# Descargar el dataset preprocesado en local

In [ ]:
import shutil
free = shutil.disk_usage('/content').free / 1e9
used = shutil.disk_usage('/content').used / 1e9
print(f'Espacio libre: {round(free, 1)} GB')
print(f'Espacio usado: {round(used, 1)} GB')

Espacio libre: 220.2 GB
Espacio usado: 22.3 GB


In [ ]:
from pathlib import Path

local = Path('/content/preprocessed_verse_for_training')
if local.exists():
    files = list(local.rglob('*'))
    print(f'✓ Copia local existe: {len(files)} archivos')
else:
    print('✗ Copia local no existe — hay que volver a copiar desde Drive')

✗ Copia local no existe — hay que volver a copiar desde Drive


In [7]:
import shutil
import os
from pathlib import Path
from tqdm.notebook import tqdm
import subprocess
import threading
from google.colab import files

DRIVE_BASE = '/content/drive/MyDrive/VerSe_2020_Dataset/preprocessed_verse_for_training'
LOCAL_COPY = '/content/preprocessed_verse_for_training'
ZIP_PATH   = '/content/preprocessed_verse_for_training.zip'

# ── 1. Copiar Drive → local ──────────────────────────────────────────
print('Copiando desde Drive a local...')
all_files  = [f for f in Path(DRIVE_BASE).rglob('*') if f.is_file()]
total_size = sum(f.stat().st_size for f in all_files)
total_gb   = round(total_size / 1e9, 2)
print(f'Total: {len(all_files)} archivos ({total_gb} GB)\n')

Path(LOCAL_COPY).mkdir(parents=True, exist_ok=True)

with tqdm(total=total_size, unit='B', unit_scale=True, desc='Copiando',
          bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]') as pbar:
    for src_file in all_files:
        rel_path = src_file.relative_to(DRIVE_BASE)
        dst_file = Path(LOCAL_COPY) / rel_path
        dst_file.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(str(src_file), str(dst_file))
        pbar.update(src_file.stat().st_size)

print('✓ Copia local completada\n')

Copiando desde Drive a local...
Total: 2841 archivos (67.19 GB)



Copiando:   0%|          | 0.00/67.2G [00:00<?, ?B/s]

✓ Copia local completada



In [11]:
import subprocess
import threading
from pathlib import Path
from tqdm.notebook import tqdm
from google.colab import files

LOCAL_COPY = '/content/preprocessed_verse_for_training'
ZIP_PATH   = '/content/preprocessed_verse_for_training.zip'

# Borrar ZIP anterior si existe
if Path(ZIP_PATH).exists():
    Path(ZIP_PATH).unlink()
    print('✓ ZIP anterior eliminado')

# Contar archivos totales
all_files   = [f for f in Path(LOCAL_COPY).rglob('*') if f.is_file()]
total_files = len(all_files)
total_size  = sum(f.stat().st_size for f in all_files)
total_gb    = round(total_size / 1e9, 2)
print(f'Comprimiendo: {total_files} archivos ({total_gb} GB)\n')

stop_flag = [False]

def monitor_zip(pbar):
    while not stop_flag[0]:
        if Path(ZIP_PATH).exists():
            size = Path(ZIP_PATH).stat().st_size
            pbar.n = min(size, total_size)
            pbar.refresh()
        threading.Event().wait(2.0)

print('Creando ZIP...')
with tqdm(total=total_size, unit='B', unit_scale=True, desc='Comprimiendo',
          bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]') as pbar:

    t = threading.Thread(target=monitor_zip, args=(pbar,))
    t.start()

    result = subprocess.run(
        ['zip', '-r', '-0', ZIP_PATH, '.'],
        cwd=LOCAL_COPY,
        capture_output=True, text=True
    )

    stop_flag[0] = True
    t.join()
    pbar.n = total_size
    pbar.refresh()

if result.returncode == 0:
    zip_gb = round(Path(ZIP_PATH).stat().st_size / 1e9, 2)
    print(f'\n✓ ZIP creado: {zip_gb} GB')
    files.download(ZIP_PATH)
else:
    print(f'✗ Error: {result.stderr[:500]}')

Comprimiendo: 2841 archivos (67.19 GB)

Creando ZIP...


Comprimiendo:   0%|          | 0.00/67.2G [00:00<?, ?B/s]


✓ ZIP creado: 67.19 GB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>